In [1]:
import sys, os, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (mean_absolute_error, root_mean_squared_error, r2_score, precision_recall_curve,
                              classification_report, confusion_matrix)
sys.path.append(os.path.abspath('..'))
from scripts.pipeline import data_engineering, classification_engineering

In [2]:
# The trained spline model is pulled to use in test and to not allow data leakage
train_spline = joblib.load('../models/training_spline_transformer.joblib')
df, test_spline = data_engineering('../data/testing_data.csv', train_spline)

In [3]:
y_test = df['Hydrogen_yield(kg)']
poly_features = ['Windspeed(m/s)' , 'GHI(W/m2)']
X_test_poly = df[poly_features]

# Pulling the trained models
poly_trans = joblib.load('../models/poly_trans.joblib')
poly_reg_model = joblib.load('../models/poly_reg_model.joblib')

X_test_poly_trans = poly_trans.transform(X_test_poly)

y_pred_poly = poly_reg_model.predict(X_test_poly_trans)

In [4]:
# Performance for the 2025 data
r2 = r2_score(y_test, y_pred_poly)
rmse = root_mean_squared_error(y_test, y_pred_poly)
n = X_test_poly_trans.shape[0]
p = X_test_poly_trans.shape[1]
adjusted_r2 = 1 - ((1 - r2) * (n - 1) / (n - p - 1))

print("2025 TEST DATA RESULTS (Polynomial Regression)")
print(f"RMSE:      {rmse:.2f} kg")
print(f"R² Score:  {r2:.4f}")
print(f"Adjusted R2 Score : {adjusted_r2:.4f}")

2025 TEST DATA RESULTS (Polynomial Regression)
RMSE:      19.65 kg
R² Score:  0.8814
Adjusted R2 Score : 0.8812


In [5]:
# Testing Random Forest over unseen data
features = ['GHI(W/m2)', 'Windspeed(m/s)', 'Stored Energy(MWh)', 'Mon',
            'Day', 'spline_hr_1','spline_hr_2', 'spline_hr_3',
            'spline_hr_4', 'Windspeed_mean_3h','Windspeed_std_3h',
            'GHI_mean_3h', 'Windspeed_lag_1hr','GHI_lag_1hr'
            ]
# Loading the model we saved from our script
rf_model = joblib.load('../models/random_forest_regressor.joblib')

X_test_rf = df[features]
y_pred_rf = rf_model.predict(X_test_rf)

rmse_rf = root_mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)
n_rf = X_test_rf.shape[0]
p_rf = X_test_rf.shape[1]
# Checking adjusted r2 to see if any of the features used are noise
adjusted_r2_rf = 1 - ((1 - r2_rf) * (n_rf - 1) / (n_rf - p_rf - 1))

print('2025 TEST DATA RESULTS (Random Forest)')
print(f"RMSE:      {rmse_rf:.2f} kg")
print(f"R² Score:  {r2_rf:.4f}")
print(f"Adjusted R2 Score : {adjusted_r2_rf:.4f}")

2025 TEST DATA RESULTS (Random Forest)
RMSE:      4.51 kg
R² Score:  0.9938
Adjusted R2 Score : 0.9937


In [6]:
y_test_shutdown = (y_test == 0).astype(int)  # 1 = Shutdown, 0 = Active

# Load SVM model and Standard scaler
svm = joblib.load('../models/svm_shutdown_classification.joblib')
scaler = joblib.load('../models/classi_scaler.joblib')

# Transform test features for SVM
X_test_scaled, _, _, _ = classification_engineering('../data/testing_data.csv', scaler=scaler, spline=train_spline)

In [7]:
decision_scores = svm.decision_function(X_test_scaled)

chosen_threshold = joblib.load('../models/svm_threshold.joblib')

# Making our binary target 
predicted_shutdown = (decision_scores >= chosen_threshold).astype(int)

# Combining out SVM model and RF
y_pred_combine = np.where(predicted_shutdown == 1, 0.0, y_pred_rf)

In [8]:
# Metrics for all hours
mae_full = mean_absolute_error(y_test, y_pred_combine)
rmse_full = root_mean_squared_error(y_test, y_pred_combine)
r2_full = r2_score(y_test, y_pred_combine)

# Metrics for hours where electrolyzer is running 
active = y_test > 0
mae_partial = mean_absolute_error(y_test[active], y_pred_combine[active])
rmse_partial = root_mean_squared_error(y_test[active], y_pred_combine[active])
r2_partial = r2_score(y_test[active], y_pred_combine[active])

print(f'SVM Threshold calculated : {chosen_threshold:.4f}')
print("Classification Report")
print(classification_report(y_test_shutdown, predicted_shutdown, target_names=['Safe Operation (0)' , 'Shutdown (1)']))

print("Confusion Matrix")
print(confusion_matrix(y_test_shutdown, predicted_shutdown))

print("Metrics for all hours:")
print(f"Complete MAE: {mae_full:.3f} kg")
print(f"Complete RMSE: {rmse_full:.3f} kg")
print(f"Complete R2: {r2_full:.4f}")\

print()

print("Metrics for hours when electrolyzer runs:")
print(f"Active MAE: {mae_partial:.3f} kg")
print(f"Active RMSE: {rmse_partial:.3f} kg")
print(f"Active R2: {r2_partial:.4f}")

SVM Threshold calculated : 0.1976
Classification Report
                    precision    recall  f1-score   support

Safe Operation (0)       1.00      1.00      1.00      7992
      Shutdown (1)       0.97      1.00      0.98       768

          accuracy                           1.00      8760
         macro avg       0.99      1.00      0.99      8760
      weighted avg       1.00      1.00      1.00      8760

Confusion Matrix
[[7971   21]
 [   3  765]]
Metrics for all hours:
Complete MAE: 1.359 kg
Complete RMSE: 4.746 kg
Complete R2: 0.9931

Metrics for hours when electrolyzer runs:
Active MAE: 1.475 kg
Active RMSE: 4.902 kg
Active R2: 0.9896
